In [1]:
import os
import glob

# 1) Disable XNNPACK BEFORE importing TF (debug only)
os.environ["TF_TFLITE_USE_XNNPACK"] = "0"

import numpy as np
import matplotlib.pyplot as plt
import cv2
from tflite_inference_debug import TFLiteInference
from tqdm import tqdm

%load_ext autoreload
%autoreload 2

2025-11-04 17:02:09.415988: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-04 17:02:09.431899: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-04 17:02:09.436761: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-04 17:02:09.449013: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-04 17:02:10.339040: W tensorflow/compiler/tf2

In [2]:
MODEL_FP32_PTH = "/home/klymy/projects/researches/people_detection_pg2/results/cam3_revB_batch1_7_margin_15_yolov8n_bg_wb_alif_clahe_box_01_dfl_01_2nd/weights/best_non_separated_saved_model_full_npu/best_float32.tflite"
MODEL_UINT8_PTH = "/home/klymy/projects/researches/people_detection_pg2/results/cam3_revB_batch1_7_margin_15_yolov8n_bg_wb_alif_clahe_box_01_dfl_01_2nd/weights/best_non_separated_saved_model_full_npu/best_full_integer_quant.tflite"

# BASE_PTH = "/home/klymy/data/people_detection/filtered_v3_margin_15/images"
BASE_PTH = "/home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/images"
IMGSZ = 320

CALIBTARION_N = 100

In [3]:
# Initialize inference class with layer indices to collect
# You can specify additional layer indices to collect during inference
layer_indices = []  # Add layer indices here if needed, e.g., [100, 200, 300]

fp32_inference = TFLiteInference(MODEL_FP32_PTH, is_uint8=False, imgsz=IMGSZ, layer_indices=layer_indices)
uint8_inference = TFLiteInference(MODEL_UINT8_PTH, is_uint8=True, imgsz=IMGSZ, layer_indices=layer_indices)

In [4]:
# Load and run inference on first image
train_img_pths = glob.glob(os.path.join(BASE_PTH, "train/*.jpg"))
print(f"Found {len(train_img_pths)} images")

val_img_pths = glob.glob(os.path.join(BASE_PTH, "val/*.jpg"))
print(f"Found {len(val_img_pths)} images")

test_img_pths = glob.glob(os.path.join(BASE_PTH, "holdout/*.jpg"))
print(f"Found {len(test_img_pths)} images")

# img = cv2.imread(img_pths[0])
# img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

np.random.seed(42)
np.random.shuffle(train_img_pths)
np.random.shuffle(val_img_pths)
np.random.shuffle(test_img_pths)
img_pths = train_img_pths[:CALIBTARION_N] + val_img_pths[:CALIBTARION_N] + test_img_pths[:CALIBTARION_N]

print(f"Total images for calibration: {len(img_pths)}")

Found 2763 images
Found 303 images
Found 148 images
Total images for calibration: 300


In [5]:
buf_fp32_boxes = []
buf_fp32_scores = []
buf_uint8_boxes = []
buf_uint8_scores = []

for img_pth in tqdm(img_pths):
    img = cv2.imread(img_pth)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    fp32_output = fp32_inference.infer_image(img.astype(np.float32) / 255)
    uint8_output = uint8_inference.infer_image(img.astype(np.uint8))
    
    uint8_scores = uint8_output["outputs"][0]
    fp32_scores = fp32_output["outputs"][1]

    uint8_coordinates = uint8_output["outputs"][1]
    fp32_coordinates = fp32_output["outputs"][0]

    idxs = np.where(uint8_scores[0] > 0.2)[0]
    buf_fp32_scores.append(fp32_scores[0, idxs])
    buf_fp32_boxes.append(fp32_coordinates[0, idxs])
    buf_uint8_scores.append(uint8_scores[0, idxs])
    buf_uint8_boxes.append(uint8_coordinates[0, idxs])


buf_fp32_scores = np.concatenate(buf_fp32_scores)
buf_fp32_boxes = np.concatenate(buf_fp32_boxes)
buf_uint8_scores = np.concatenate(buf_uint8_scores)
buf_uint8_boxes = np.concatenate(buf_uint8_boxes)

print("FP32 scores shape:", buf_fp32_scores.shape)
print("FP32 boxes shape:", buf_fp32_boxes.shape)
print("UINT8 scores shape:", buf_uint8_scores.shape)
print("UINT8 boxes shape:", buf_uint8_boxes.shape)

folder_name = "wb_alif_clahe_collected_outputs_nano"
os.makedirs(folder_name, exist_ok=True)
np.save(f"{folder_name}/fp32_scores.npy", buf_fp32_scores)
np.save(f"{folder_name}/fp32_boxes.npy", buf_fp32_boxes)
np.save(f"{folder_name}/uint8_scores.npy", buf_uint8_scores)
np.save(f"{folder_name}/uint8_boxes.npy", buf_uint8_boxes)

  0%|          | 0/300 [00:00<?, ?it/s]

100%|██████████| 300/300 [05:26<00:00,  1.09s/it]

FP32 scores shape: (12299, 1)
FP32 boxes shape: (12299, 6)
UINT8 scores shape: (12299, 1)
UINT8 boxes shape: (12299, 6)
